# SpliceAImod — run the shard driver from a browser session

Minimal runner: everything happens in `examples/colab_run.py` (install, reference, pre-cut shards,
driver launch). Runtime → Change runtime type → **A100**. Run the cells top to bottom, then leave
the tab open (Pro) or close it (Pro+ background execution). Open this same notebook in several
runtimes to add sessions; they share shards through Drive claims.

Edit only the `os.environ[...]` lines in cell 2 if your Drive layout differs from
`MyDrive/spliceai_run/{shards,ref,out,logs}`.

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()     # only needed when GCS_ROOT is set below; harmless otherwise

In [ ]:
import os, subprocess, runpy
os.environ.update({
    "INSTALL": "1",
    "COLAB_AUTO_UNASSIGN": "1",              # driver releases this runtime when nothing is left to claim
    # "GCS_ROOT": "gs://my-bucket/spliceai_run",   # shards from $GCS_ROOT/shards/, results to $GCS_ROOT/out/
    # "SHARDS_URL": "https://github.com/chundruv/SpliceAImod/releases/download/shards-v1",  # shards served from a GitHub release instead of Drive
    # "DRIVE_ROOT": "/content/drive/MyDrive/spliceai_run",
    # "ANNOTATION": "gencodev49",           # or MANEv1.4
    # "EXTRA_FLAGS": "--compile --conv-impl valid_nhwc",
    # "TORCH_BATCH": "256", "PRED_BATCH": "8192", "BATCH_WORKERS": "4",
})
if not os.path.isdir("/content/SpliceAImod"):
    subprocess.run(["git", "clone", "-q", "https://github.com/chundruv/SpliceAImod.git", "/content/SpliceAImod"], check=True)
_ = runpy.run_path("/content/SpliceAImod/examples/colab_run.py", run_name="__main__"); del _

## Monitor (re-run any time)

In [ ]:
import glob, subprocess, json
logs = sorted(glob.glob("/content/drive/MyDrive/spliceai_run/logs/driver_*.log"))
print(open(logs[-1]).read()[-2500:] if logs else "no log yet")
print("--- gpu worker:", *subprocess.run("tail -n 4 /content/work/tmp/*/GPU_0_w0.stderr 2>/dev/null", shell=True, capture_output=True, text=True).stdout.splitlines()[-4:], sep="\n")
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!pgrep -af "python[0-9.]* /content/[c]olab_driver.py" || echo "driver not running"
done = len(glob.glob("/content/drive/MyDrive/spliceai_run/out/shard_*.done")); claims = len(glob.glob("/content/drive/MyDrive/spliceai_run/out/shard_*.claim"))
print(f"{done} shards finished on Drive, {claims} claimed by running sessions")

## Optional: block until this session's driver exits, then release the runtime

Use on Pro+ (background execution) or with the tab left open. The driver already releases the
runtime itself via `COLAB_AUTO_UNASSIGN`; this is the belt-and-braces path.

In [ ]:
import subprocess, time
while subprocess.run(["pgrep", "-f", "python[0-9.]* /content/[c]olab_driver.py"], capture_output=True).returncode == 0:
    time.sleep(60)
from google.colab import runtime; runtime.unassign()